# Tier 3 — ajuste final do FT-CUR (só isso)

Seleção `per_batch` lida do **repositório** (`results/tier3_selection_cur_per_gpu*.json`), não de entrada.
Sem célula de seleção: se algo estiver errado, a asserção para em segundos.

- Acelerador: **GPU T4 ×2**. *Save Version → Save & Run All*. **Não cancele**: o prazo de 8 h encerra sozinho e salva `tier3_out/`.
- Retomada: adicione a saída de uma sessão anterior **deste** notebook como entrada (só `tier3_final_cur_*` é retomado).
- A saída é só `tier3_out/` (KB): o clone fica em `/tmp`.
- O progresso de cada semente aparece no log ao vivo.

In [ ]:
# ── 1. GPUs ──
import torch
print('CUDA:', torch.cuda.is_available(), '| placas:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()): print(' ', i, torch.cuda.get_device_name(i))
assert torch.cuda.device_count() >= 2, 'Settings → Accelerator → GPU T4 x2'

In [ ]:
# ── 2. Repositório ──
import os, subprocess, sys
GIT_URL     = 'https://github.com/PauloBernardo/dissertacao-estudo-comparativo.git'
# clone FORA de /kaggle/working: tudo lá vira saída (parquets + .git = centenas de MB → "adicionando dados"
# lento como entrada e salvamento que pode falhar). A saída fica só com tier3_out/.
PROJECT_DIR = '/tmp/sparse-lssvm-transformers-study'
BRANCH      = 'revisao/estatistica-e-proveniencia'
if os.path.exists(PROJECT_DIR):
    subprocess.run(['git', '-C', PROJECT_DIR, 'fetch', 'origin'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, GIT_URL, PROJECT_DIR], check=True)
subprocess.run(['git', '-C', PROJECT_DIR, 'checkout', BRANCH], check=True)
subprocess.run(['git', '-C', PROJECT_DIR, 'pull', '--rebase', 'origin', BRANCH], check=True)
os.chdir(PROJECT_DIR)
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)
sys.path.insert(0, '.')
assert 'TIER3_DEADLINE' in open('scripts/run_tier3.py').read(), 'clone sem prazo no runner — refaça o pull'

In [ ]:
# ── 3. Dependências e dados (em cache no clone; nada é baixado) ──
subprocess.run('pip install -q einops optuna', shell=True)
from pathlib import Path
from src.data.loaders import DatasetLoader
for d in ['HIGGS500K', 'COVER', 'SUSY']:
    assert Path(f'data/raw/{d}.parquet').exists(), f'sem {d}.parquet no clone — refaça o pull'
    X, y, _ = DatasetLoader.load(d)
    print(f'{d:10s} {X.shape}  positivos {y.mean():.3f}')

In [ ]:
# ── 4. Configuração ──
import json, shutil, threading, time
from pathlib import Path
DEADLINE_H = 8.0     # termina sozinho com folga: o Kaggle só salva a saída se a sessão terminar
os.environ['TIER3_DEADLINE'] = str(time.time() + DEADLINE_H * 3600)
OUTDIR = Path('/kaggle/working/tier3_out'); OUTDIR.mkdir(exist_ok=True)
CUR_SPLIT = {0: 'HIGGS500K SUSY', 1: 'COVER'}
# seleção per_batch: SEMPRE a do repositório (nunca de entrada)
for g in (0, 1):
    shutil.copy(f'results/tier3_selection_cur_per_gpu{g}.json', OUTDIR / f'tier3_selection_cur_gpu{g}.json')
sel = {**json.load(open(OUTDIR / 'tier3_selection_cur_gpu0.json'))['FTTransformerCURRandom'],
       **json.load(open(OUTDIR / 'tier3_selection_cur_gpu1.json'))['FTTransformerCURRandom']}
assert all(sel[d]['complete'] for d in ('HIGGS500K', 'SUSY', 'COVER'))
assert abs(sel['HIGGS500K']['params']['lr'] - 0.0014691510255403506) < 1e-12, 'seleção errada (não é a per_batch)'
assert sel['COVER']['params'] == {'n_layers': 3, 'n_heads': 4, 'lr': 0.0006557847832445657}, 'seleção errada'
for d, r in sel.items(): print(f'seleção {d:10s} F1val {r["best_val_f1"]:.4f} {r["params"]}')
# retomada: só resultados finais de uma sessão anterior deste notebook
for f in sorted(Path('/kaggle/input').rglob('tier3_final_cur_gpu*')):
    if f.is_file() and not (OUTDIR / f.name).exists():
        shutil.copy(f, OUTDIR / f.name); print('retomado:', f)

def files(gpu):
    return (f'--selection-file {OUTDIR}/tier3_selection_cur_gpu{gpu}.json '
            f'--final-file {OUTDIR}/tier3_final_cur_gpu{gpu}.json')

def run_2gpu(make_cmd, name):
    """Um processo por placa; cada linha vai para o log em tier3_out/ E para a tela."""
    def pump(gpu, proc, log):
        for line in proc.stdout:
            log.write(line); log.flush(); print(f'[gpu{gpu}] {line}', end='', flush=True)
    procs = []
    for gpu in (0, 1):
        log = open(OUTDIR / f'tier3_{name}_gpu{gpu}.log', 'a')
        cmd = f'CUDA_VISIBLE_DEVICES={gpu} python -u ' + make_cmd(gpu)
        print(f'[gpu{gpu}]', cmd, flush=True)
        p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        t = threading.Thread(target=pump, args=(gpu, p, log)); t.start(); procs.append((gpu, p, t, log))
    for g, p, t, log in procs:
        p.wait(); t.join(); log.close(); print(f'gpu{g} terminou com código {p.returncode}')

In [ ]:
# ── 5. Ajuste final do FT-CUR (10 sementes) ──
run_2gpu(lambda g: 'scripts/run_tier3.py final --variants FTTransformerCURRandom --datasets ' + CUR_SPLIT[g]
         + ' --seeds 0 1 2 3 4 5 6 7 8 9 ' + files(g), 'final_cur')

In [ ]:
# ── 6. Situação ──
import collections, statistics as stt
recs = [r for g in (0, 1) if (OUTDIR / f'tier3_final_cur_gpu{g}.json').exists()
        for r in json.load(open(OUTDIR / f'tier3_final_cur_gpu{g}.json'))]
by = collections.defaultdict(list)
for r in recs:
    if r.get('status') == 'ok': by[r['dataset']].append(r['test_f1_macro'])
falta = 0
for d in ('HIGGS500K', 'COVER', 'SUSY'):
    xs = by[d]; falta += 10 - len(xs)
    print(f'FT-CUR {d:10s} {len(xs):2d}/10' + (f'  F1 {stt.mean(xs):.4f} ± {stt.pstdev(xs):.4f}' if xs else ''))
print([(r['dataset'], r['seed'], r['error'][:80]) for r in recs if r.get('status') != 'ok'][:5])
print('FINAL COMPLETO — baixe tier3_out/.' if not falta else f'{falta} pendentes — nova versão com esta saída como entrada.')